# Black-Scholes Options Pricing: Walkthrough

Step by step: price an option, read its Greeks, back out implied volatility, then test the model against real SPX quotes.

In [1]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, '../src')
from options_pricing import black_scholes as bs, market
pd.set_option('display.precision', 4)

## 1. Price a call and a put
A textbook check (Hull): S = 42, K = 40, r = 10%, σ = 20%, six months. Expected: call 4.76, put 0.81.

In [2]:
for kind in ('call', 'put'):
    print(kind, round(bs.price(S=42, K=40, T=0.5, r=0.10, sigma=0.20, kind=kind), 4))

call 4.7594
put 0.8086


## 2. Put-call parity
C − P must equal S − K·e^(−rT). If it didn't, there would be a riskless profit.

In [3]:
c, p = bs.price(42, 40, 0.5, 0.10, 0.20, 'call'), bs.price(42, 40, 0.5, 0.10, 0.20, 'put')
print(c - p, 42 - 40 * np.exp(-0.10 * 0.5))

3.9508230199714394 3.9508230199714376


## 3. The Greeks
An at-the-money SPX-sized option: S = K = 7,200, 24 days, σ = 24%, r = 4.2%.

In [4]:
pd.DataFrame({k: bs.greeks(7200, 7200, 24/365, 0.042, 0.24, k) for k in ('call', 'put')})

,call,put
delta,0.5301,-0.4699
gamma,0.0009,0.0009
vega,7.3444,7.3444
theta,-4.0900,-3.2638
rho,2.3872,-2.3340


Reading it: a $1 index move changes the call by about $0.53 (delta); one vol point is worth about $7.34 (vega); each day costs the call holder about $4.09 (theta).

## 4. Implied volatility
Invert the model: which σ reproduces a given price?

In [5]:
px = bs.price(7200, 7200, 24/365, 0.042, 0.24, 'call')
bs.implied_vol(px, 7200, 7200, 24/365, 0.042, 'call')

0.24000000000000063

## 5. Real market: SPX options expiring 2026-12-18

In [6]:
chain = market.clean_chain(market.load_chain('../data/' + market.SNAPSHOT_FILE))
T, r = market.year_fraction('2026-11-24', '2026-12-18'), 0.042
fwd = market.parity_forward(chain, market.SNAPSHOT_SPOT, T, r)
s_eff = fwd * np.exp(-r * T)
print(f'forward {fwd:,.2f}   parity-implied spot {s_eff:,.2f}   index close {market.SNAPSHOT_SPOT:,.2f}')
print('parity-implied r:', round(market.parity_rates(chain, market.SNAPSHOT_SPOT, T)[0], 4))

forward 7,155.72   parity-implied spot 7,135.98   index close 7,201.09
parity-implied r: 0.0472


In [7]:
smile = market.otm_smile(market.add_implied_vols(chain, s_eff, T, r, 0.0), fwd)
atm = market.atm_vol(smile, fwd)
print(f'ATM implied vol {atm:.2%}')
smile[['type', 'strike', 'mid', 'iv']].iloc[::4]

ATM implied vol 24.12%


,type,strike,mid,iv
0,put,6000,16.750,0.4296
4,put,6200,24.100,0.3961
8,put,6400,34.700,0.3613
12,put,6600,50.750,0.3260
16,put,6800,78.500,0.2950
20,put,7000,122.700,0.2637
24,call,7200,150.350,0.2343
28,call,7400,56.950,0.1993
32,call,7600,13.500,0.1743
36,call,7800,2.875,0.1695


## 6. One volatility for every strike?
Black-Scholes says yes. The market says no.

In [8]:
comp = market.compare_flat_vol(smile, s_eff, T, r, 0.0, atm)
comp[['type', 'strike', 'mid', 'bs_price', 'error_pct']].iloc[::4]

,type,strike,mid,bs_price,error_pct
0,put,6000,16.750,0.2615,-0.9844
4,put,6200,24.100,1.4270,-0.9408
8,put,6400,34.700,5.8890,-0.8303
12,put,6600,50.750,19.0057,-0.6255
16,put,6800,78.500,49.5212,-0.3692
20,put,7000,122.700,107.3928,-0.1248
24,call,7200,150.350,155.3959,0.0336
28,call,7400,56.950,82.9854,0.4572
32,call,7600,13.500,39.8078,1.9487
36,call,7800,2.875,17.1215,4.9553


Low-strike puts trade far above the flat-vol price (crash protection is in demand), and high-strike calls trade far below it. That pattern is the **volatility skew**, and it is the main reason practitioners extend Black-Scholes with smile, stochastic-volatility and jump models.